# Pasokin — Eval Runner (Triage / Parser / Optimizer)

## 1. Config

In [ ]:
LABEL = "baseline"                  
RUN_COMPONENTS = ["optimizer", "parser", "triage"]   

REPO_URL = "https://github.com/DavidLauda/Pasokin.git"   
BRANCH = "main"

TRIAGE_MODE = "local"
TRIAGE_URL = "http://localhost:8001"
TRIAGE_TIMEOUT_S = 120

PARSER_SLEEP_S = 1.5        # pause between Gemini calls (rate limits trigger the silent heuristic fallback)
OVERWRITE = False           # refuse to overwrite existing results files for the same LABEL

# Unit aliases for the parser check -- decide BEFORE the baseline, then don't change them mid-day
UNIT_ALIASES = {"l": "liter", "ltr": "liter", "litre": "liter", "pieces": "pcs", "piece": "pcs",
                "kardus": "dus", "box": "dus"}

## 2. Secrets


In [ ]:
import os
from google.colab import userdata

def secret(name, required=True):
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise RuntimeError(f"Secret {name} missing: add it in the key-icon panel and enable notebook access")
        return None

os.environ["GEMINI_API_KEY"] = secret("GEMINI_API_KEY", required="parser" in RUN_COMPONENTS)
os.environ["HF_TOKEN"] = secret("HF_TOKEN", required=("triage" in RUN_COMPONENTS and TRIAGE_MODE == "local")) or ""
print("secrets loaded")

## 3. Get the code


In [ ]:
import subprocess
REPO_DIR = "/content/Pasokin"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

GIT_SHA = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
BACKEND_DIR = f"{REPO_DIR}/backend"
EVAL_DIR = f"{BACKEND_DIR}/eval"
CASES_DIR = f"{EVAL_DIR}/cases"
RESULTS_DIR = f"{EVAL_DIR}/results"
WRAPPER_DIR = f"{EVAL_DIR}/wrappers"
for d in (CASES_DIR, RESULTS_DIR, WRAPPER_DIR):
    os.makedirs(d, exist_ok=True)
print("commit:", GIT_SHA)

## 4. Node.js + backend packages

In [ ]:
v = subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip()
print("node:", v or "not installed")
if not v or int(v.lstrip("v").split(".")[0]) < 20:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null
    !apt-get install -y nodejs > /dev/null
    print("node now:", subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip())
!cd {BACKEND_DIR} && npm install --silent
print("backend packages installed")

## 5. Test cases

In [ ]:
import shutil, json
missing = [c for c in RUN_COMPONENTS if not os.path.exists(f"{CASES_DIR}/{c}.json")]
if missing:
    from google.colab import files
    print("Upload the case files for:", missing)
    for name in files.upload():
        for comp in missing:
            if comp in name.lower():
                shutil.move(name, f"{CASES_DIR}/{comp}.json")
                print(f"{name} -> cases/{comp}.json")
for c in RUN_COMPONENTS:
    path = f"{CASES_DIR}/{c}.json"
    if not os.path.exists(path):
        raise FileNotFoundError(f"no cases for {c}: {path}")
    print(c, "cases:", len(json.load(open(path, encoding="utf-8"))), path)

## 6. Start the triage service 

In [ ]:
import time, requests
if "triage" in RUN_COMPONENTS and TRIAGE_MODE == "local":
    !pip -q install fastapi uvicorn peft accelerate bitsandbytes python-dotenv
    subprocess.run(["pkill", "-f", "uvicorn main:app"])      # stop an old instance (new adapter / code)
    env = dict(os.environ, DEMO_MODE="false", USE_4BIT="auto", ADAPTER_PATH="./adapter_v2") #untuk ngeload model ke 2
    log = open("/content/triage_service.log", "w")
    subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8001"],
                     cwd=f"{REPO_DIR}/triage-service", env=env, stdout=log, stderr=log)
    t0 = time.time()
    while True:
        try:
            h = requests.get(f"{TRIAGE_URL}/health", timeout=5).json()
            if h.get("model_loaded"):
                print(f"triage service ready ({time.time() - t0:.0f}s):", h)
                break
        except Exception:
            pass
        if time.time() - t0 > 900:
            raise RuntimeError("triage service not ready after 15 min -- see /content/triage_service.log")
        time.sleep(10)

if "triage" in RUN_COMPONENTS:
    h = requests.get(f"{TRIAGE_URL}/health", timeout=10).json()
    assert h.get("model_loaded"), f"model not loaded (DEMO_MODE on?): {h}"
    print("health:", h)

## 7. Node wrappers 

In [ ]:
%%writefile {WRAPPER_DIR}/parser_wrapper.js
// Input (stdin):  {"input": "...", "message_date": "YYYY-MM-DD"}
// Output (stdout): {"result": <parseRequirementIntent output>, "method": "gemini" | "heuristic" | "error"}
const path = require('path');

// Keep stdout clean for the JSON result: all logs go to stderr (set up BEFORE loading anything).
const toErr = (...a) => process.stderr.write(a.map(String).join(' ') + '\n');
let fellBack = false;
console.log = toErr; console.info = toErr; console.warn = toErr;
console.error = (...a) => { if (a.some(x => String(x).includes('falling back'))) fellBack = true; toErr(...a); };
try { require('dotenv').config({ path: path.join(__dirname, '..', '..', '.env'), quiet: true }); } catch (_) {}

let raw = '';
process.stdin.on('data', d => (raw += d));
process.stdin.on('end', async () => {
  const c = JSON.parse(raw);
  // Freeze "today" at the case's message_date (09:00 WIB) so relative dates
  // ("besok", "rabu depan", "tanggal 30") are reproducible on any run day.
  const RealDate = Date;
  const fixed = new RealDate(`${c.message_date}T09:00:00+07:00`).getTime();
  class FixedDate extends RealDate {
    constructor(...args) { if (args.length === 0) super(fixed); else super(...args); }
    static now() { return fixed; }
  }
  global.Date = FixedDate;

  const { parseRequirementIntent } = require('../../src/services/geminiService');
  try {
    const result = await parseRequirementIntent(c.input);
    process.stdout.write(JSON.stringify({ result, method: fellBack ? 'heuristic' : 'gemini' }));
  } catch (e) {
    process.stdout.write(JSON.stringify({ result: null, method: 'error', error: e.message }));
  }
});

In [ ]:
%%writefile {WRAPPER_DIR}/optimizer_wrapper.js
// Input (stdin):  one optimizer test case (demand_qty, target_kirim_days, weights 0-1, suppliers[...])
// Output (stdout): {"allocation": {name: qty>0}, "returned_null": bool, "total_allocated": n, "rows": [...]}
const toErr = (...a) => process.stderr.write(a.map(String).join(' ') + '\n');
console.log = toErr; console.info = toErr; console.warn = toErr;

let raw = '';
process.stdin.on('data', d => (raw += d));
process.stdin.on('end', () => {
  const c = JSON.parse(raw);
  const { optimizeAllocation } = require('../../src/services/optimizerService');
  const w = c.weights || {};
  const requirement = {
    quantity: c.demand_qty,
    maxBudget: c.max_budget,                 // undefined -> no budget limit
    maxLeadTimeDays: c.target_kirim_days,    // unused by the current code; picked up once a deadline filter exists
    priority: { cost: (w.cost ?? 0) * 100, speed: (w.speed ?? 0) * 100, risk: (w.reliability ?? 0) * 100 }
  };
  const candidates = (c.suppliers || []).map(s => ({
    id: s.name, name: s.name,
    price_per_unit: s.price,
    max_capacity_qty: s.qty,
    lead_time_days: s.lead_time_days,
    reliability_score: s.reliability_score,
    min_order_qty: s.moq ?? s.min_order_qty ?? 0
  }));
  const out = optimizeAllocation(requirement, candidates);
  const rows = out?.recommended_allocations || [];
  const allocation = Object.fromEntries(rows.filter(r => r.qty > 0).map(r => [r.name, r.qty]));
  process.stdout.write(JSON.stringify({
    allocation, returned_null: out == null,
    total_allocated: rows.reduce((s, r) => s + (r.qty || 0), 0), rows
  }));
});

## 8. Harness (metrics + result files) 

In [ ]:
import json, os, re, time, datetime, subprocess
from collections import Counter
import requests

# ---------- helpers ----------
def to_num(v):
    """Numbers and plain numeric strings -> float. Anything else never equals a number."""
    if v is None or isinstance(v, bool):
        return None if v is None else ("RAW", v)
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str) and re.fullmatch(r"-?\d+(\.\d+)?", v.strip()):
        return float(v.strip())
    return ("RAW", v)

def num_eq(a, b):
    return to_num(a) == to_num(b)

def norm_text(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower() if s is not None else None

def norm_unit(u):
    u = norm_text(u)
    return UNIT_ALIASES.get(u, u) if u is not None else None

def run_node(script, payload, timeout=90):
    try:
        p = subprocess.run(["node", script], input=json.dumps(payload, ensure_ascii=False),
                           capture_output=True, text=True, timeout=timeout, cwd=BACKEND_DIR, env=os.environ)
    except subprocess.TimeoutExpired:
        return None, "", "timeout"
    lines = [l for l in p.stdout.strip().splitlines() if l.strip().startswith("{")]
    out = lines[-1] if lines else p.stdout.strip()
    try:
        return json.loads(out), p.stderr[-500:], None if p.returncode == 0 else f"exit_{p.returncode}"
    except Exception:
        return None, p.stderr[-500:], f"bad_output: {out[:200]}"

def meta(component):
    return {"component": component, "label": LABEL, "git_sha": GIT_SHA,
            "timestamp": datetime.datetime.now().isoformat(timespec="seconds")}

def rate(a, b):
    return round(a / b, 4) if b else None

# ---------- TRIAGE ----------
def call_triage(case):
    t0 = time.time()
    try:
        r = requests.post(f"{TRIAGE_URL}/triage", json={"text_input": case["input"]}, timeout=TRIAGE_TIMEOUT_S)
    except Exception as e:
        return {"parsed": None, "error_type": "service", "error": str(e)[:200], "raw": None,
                "latency_s": round(time.time() - t0, 2)}
    lat = round(time.time() - t0, 2)
    if r.status_code == 200:
        body = r.json()
        return {"parsed": body, "error_type": None, "error": None,
                "raw": body.get("raw_model_output"), "latency_s": lat}
    if r.status_code == 422:
        detail = (r.json() or {}).get("detail", {})
        raw = detail.get("raw_model_output") if isinstance(detail, dict) else None
        return {"parsed": None, "error_type": "invalid_json", "error": "model output not valid/complete JSON",
                "raw": raw, "latency_s": lat}
    return {"parsed": None, "error_type": "service", "error": f"http_{r.status_code}", "raw": r.text[:200],
            "latency_s": lat}

def eval_triage(cases):
    results = []
    for case in cases:
        r = call_triage(case)
        p = r["parsed"] or {}
        act = norm_text(p.get("classification")) if r["parsed"] else None
        exp = case["expected_classification"]
        exp_x = case.get("expected_extracted") or {}
        act_x = p.get("ai_extracted") or {}
        field_correct = {f: (num_eq(act_x.get(f), exp_x.get(f)) if r["parsed"] else False)
                         for f in ("qty", "price", "lead_time_days") if f in exp_x}
        class_pass = act == exp
        results.append({
            "case_id": case["case_id"], "tags": case.get("tags", []), "input": case["input"],
            "expected_classification": exp, "actual_classification": act,
            "class_pass": class_pass,
            "expected_extracted": exp_x, "actual_extracted": act_x,
            "field_correct": field_correct,
            "field_fails": [f for f, ok in field_correct.items() if not ok],
            "pass": class_pass and all(field_correct.values()),
            "error_type": r["error_type"], "error": r["error"],
            "latency_s": r["latency_s"], "raw_model_output": r["raw"],
        })
        print(f'{case["case_id"]:<14} {"PASS" if results[-1]["pass"] else "FAIL"}  '
              f'class={act} fields_wrong={results[-1]["field_fails"]} {r["error_type"] or ""}')

    TP = FP = FN = TN = 0
    other = Counter()
    for x in results:
        e, a = x["expected_classification"], x["actual_classification"]
        if a not in ("confirmed", "needs_manual_review"):
            other[f"expected={e}"] += 1
            if e == "confirmed":
                FN += 1                     # expected confirmed, did not get confirmed
            continue
        if e == "confirmed" and a == "confirmed": TP += 1
        elif e != "confirmed" and a == "confirmed": FP += 1
        elif e == "confirmed": FN += 1
        else: TN += 1

    n = len(results)
    field_acc = {}
    for f in ("qty", "price", "lead_time_days"):
        vals = [x["field_correct"][f] for x in results if f in x["field_correct"]]
        field_acc[f] = rate(sum(vals), len(vals))
    service_errors = sum(1 for x in results if x["error_type"] == "service")
    summary = {**meta("triage"), "triage_url": TRIAGE_URL, "n_cases": n,
        "confusion_matrix": {"TP (exp confirmed, got confirmed)": TP,
                             "FN (exp confirmed, got review/error)": FN,
                             "FP (exp review, got confirmed)": FP,
                             "TN (exp review, got review)": TN,
                             "no_valid_label_by_expected": dict(other)},
        "false_confirmed_rate": rate(FP, TP + FP),
        "missed_confirmation_rate": rate(FN, TP + FN),
        "classification_accuracy": rate(sum(x["class_pass"] for x in results), n),
        "json_validity": rate(sum(1 for x in results if x["error_type"] != "invalid_json"), n),
        "extraction_accuracy": field_acc,
        "overall_pass_rate": rate(sum(x["pass"] for x in results), n),
        "service_errors": service_errors,
        "avg_latency_s": rate(sum(x["latency_s"] for x in results), n)}
    if service_errors:
        print(f"\n[WARN] {service_errors} service errors (timeouts/connection). These are NOT model failures -- fix the service and rerun.")
    return results, summary

# ---------- PARSER ----------
def eval_parser(cases):
    results = []
    methods = Counter()
    for case in cases:
        exp = case.get("expected")
        if case.get("needs_manual_verification") or exp is None:
            results.append({"case_id": case["case_id"], "input": case["input"], "skipped": True,
                            "reason": "ambiguous / no ground truth yet"})
            continue
        out, stderr, err = run_node(f"{WRAPPER_DIR}/parser_wrapper.js",
                                    {"input": case["input"], "message_date": case["message_date"]})
        method = (out or {}).get("method", "error")
        methods[method] += 1
        act = (out or {}).get("result") or {}

        off = exp.get("targetDeliveryDate_offset_days")
        exp_date = (datetime.date.fromisoformat(case["message_date"]) + datetime.timedelta(days=off)).isoformat() \
            if off is not None else None
        act_date = str(act.get("targetDeliveryDate"))[:10] if act.get("targetDeliveryDate") else None

        checks = {
            "materialName": norm_text(act.get("materialName")) == norm_text(exp.get("materialName")),
            "quantity": num_eq(act.get("quantity"), exp.get("quantity")),
            "unit": norm_unit(act.get("unit")) == norm_unit(exp.get("unit")),
            "maxBudget": num_eq(act.get("maxBudget"), exp.get("maxBudget")),
            "targetDeliveryDate": act_date == exp_date,
        }
        if exp.get("priority"):
            ap = act.get("priority") or {}
            checks["priority"] = all(num_eq(ap.get(k), exp["priority"].get(k)) for k in ("cost", "speed", "risk"))

        results.append({"case_id": case["case_id"], "tags": case.get("tags", []), "input": case["input"],
                        "message_date": case["message_date"], "method": method,
                        "expected": exp, "expected_date": exp_date, "actual": act, "actual_date": act_date,
                        "checks": checks, "field_fails": [k for k, ok in checks.items() if not ok],
                        "pass": all(checks.values()), "error": err,
                        "stderr_tail": stderr if (err or method != "gemini") else None})
        print(f'{case["case_id"]:<12} {"PASS" if results[-1]["pass"] else "FAIL"}  [{method}] wrong={results[-1]["field_fails"]}')
        time.sleep(PARSER_SLEEP_S)

    run = [x for x in results if not x.get("skipped")]
    field_acc = {}
    for f in ("materialName", "quantity", "unit", "maxBudget", "targetDeliveryDate", "priority"):
        vals = [x["checks"][f] for x in run if f in x["checks"]]
        if vals:
            field_acc[f] = rate(sum(vals), len(vals))
    summary = {**meta("parser"), "n_cases": len(results), "cases_run": len(run),
               "skipped_ambiguous": len(results) - len(run), "method_counts": dict(methods),
               "field_accuracy": field_acc,
               "overall_pass_rate": rate(sum(x["pass"] for x in run), len(run))}
    if methods.get("heuristic") or methods.get("error"):
        print(f"\n[WARN] {methods.get('heuristic', 0)} cases ran on the HEURISTIC fallback and "
              f"{methods.get('error', 0)} errored -- these numbers are not pure Gemini. Check the key / rate limits.")
    return results, summary

# ---------- OPTIMIZER ----------
def eval_optimizer(cases):
    results = []
    viol_types = Counter()
    for case in cases:
        out, stderr, err = run_node(f"{WRAPPER_DIR}/optimizer_wrapper.js", case, timeout=30)
        if out is None:
            results.append({"case_id": case["case_id"], "type": case.get("type"), "crashed": True,
                            "error": err, "stderr_tail": stderr, "violations": ["crash"], "pass": False})
            print(f'{case["case_id"]:<16} FAIL  crash: {err}')
            continue
        alloc = {k: v for k, v in (out.get("allocation") or {}).items() if v}
        sups = {s["name"]: s for s in case.get("suppliers", [])}
        total = sum(alloc.values())
        v = []
        if total > case["demand_qty"]:
            v.append("over_demand")
        for name, q in alloc.items():
            s = sups.get(name)
            if s is None:
                v.append("unknown_supplier"); continue
            if s.get("qty") is None or q > s["qty"]:
                v.append("over_capacity")
            moq = s.get("moq", s.get("min_order_qty"))
            if moq and q < moq:
                v.append("below_moq")
            if case.get("target_kirim_days") is not None and s.get("lead_time_days", 0) > case["target_kirim_days"]:
                v.append("late")
        if case.get("max_budget") is not None:
            spent = sum(q * sups[n]["price"] for n, q in alloc.items() if n in sups)
            if spent > case["max_budget"]:
                v.append("over_budget")
        v = sorted(set(v))
        for t in v:
            viol_types[t] += 1

        feasible = total >= case["demand_qty"] and not v
        exp_alloc = case.get("expect_allocation")
        alloc_checked = exp_alloc is not None
        alloc_ok = (not alloc_checked) or \
            ({k: float(q) for k, q in alloc.items()} == {k: float(q) for k, q in exp_alloc.items() if q})
        feasible_ok = feasible == case.get("expect_feasible")
        results.append({"case_id": case["case_id"], "type": case.get("type"),
                        "expected_feasible": case.get("expect_feasible"), "actual_feasible": feasible,
                        "expected_allocation": exp_alloc, "actual_allocation": alloc,
                        "allocation_checked": alloc_checked, "returned_null": out.get("returned_null"),
                        "violations": v, "feasible_ok": feasible_ok, "allocation_ok": alloc_ok,
                        "pass": feasible_ok and alloc_ok and not v, "note": case.get("note")})
        print(f'{case["case_id"]:<16} {"PASS" if results[-1]["pass"] else "FAIL"}  alloc={alloc} violations={v}')

    n = len(results)
    ran = [x for x in results if not x.get("crashed")]
    summary = {**meta("optimizer"), "n_cases": n, "crashes": n - len(ran),
               "test_pass_rate": rate(sum(x["pass"] for x in results), n),     # crashes count as failures
               "constraint_violation_rate": rate(sum(1 for x in ran if x["violations"]), len(ran)),
               "violations_by_type": dict(viol_types),
               "allocation_checked_cases": sum(1 for x in ran if x.get("allocation_checked"))}
    return results, summary

# ---------- run ----------
EVALS = {"triage": eval_triage, "parser": eval_parser, "optimizer": eval_optimizer}

def run_component(comp):
    res_path = f"{RESULTS_DIR}/{comp}_{LABEL}.json"
    sum_path = f"{RESULTS_DIR}/{comp}_{LABEL}_summary.json"
    if os.path.exists(res_path) and not OVERWRITE:
        raise FileExistsError(f"{res_path} exists -- change LABEL (or set OVERWRITE=True on purpose)")
    cases = json.load(open(f"{CASES_DIR}/{comp}.json", encoding="utf-8"))
    print(f"\n===== {comp.upper()} ({LABEL}, {len(cases)} cases, commit {GIT_SHA}) =====")
    results, summary = EVALS[comp](cases)
    for path, obj in ((res_path, results), (sum_path, summary)):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print("->", res_path, "\n->", sum_path)
    return summary

## 9. Run

In [ ]:
SUMMARIES = {}
for comp in RUN_COMPONENTS:
    SUMMARIES[comp] = run_component(comp)

## 10. Failed cases at a glance 

In [ ]:
for comp in RUN_COMPONENTS:
    rows = json.load(open(f"{RESULTS_DIR}/{comp}_{LABEL}.json", encoding="utf-8"))
    failed = [r for r in rows if r.get("pass") is False]
    print(f"\n{comp}: {len(failed)} failed")
    for r in failed:
        why = r.get("field_fails") or r.get("violations") or r.get("error") or ""
        extra = ""
        if comp == "triage":
            extra = f'exp={r["expected_classification"]} got={r["actual_classification"]}'
        elif comp == "optimizer":
            extra = f'feasible exp={r.get("expected_feasible")} got={r.get("actual_feasible")}'
        print(f'  {r["case_id"]:<16} {extra} {why}  tags={r.get("tags", "")}')

## 11. Compare with an earlier run 

In [ ]:
BEFORE = "baseline"
for comp in RUN_COMPONENTS:
    a_path, b_path = f"{RESULTS_DIR}/{comp}_{BEFORE}.json", f"{RESULTS_DIR}/{comp}_{LABEL}.json"
    if BEFORE == LABEL or not os.path.exists(a_path):
        print(f"{comp}: no '{BEFORE}' results in this session (upload them into {RESULTS_DIR} to compare)")
        continue
    a = {r["case_id"]: r.get("pass") for r in json.load(open(a_path))}
    b = {r["case_id"]: r.get("pass") for r in json.load(open(b_path))}
    fixed = [k for k in b if a.get(k) is False and b[k] is True]
    broke = [k for k in b if a.get(k) is True and b[k] is False]
    print(f"{comp}: FAIL->PASS {fixed}\n{' ' * len(comp)}  PASS->FAIL (regressions!) {broke}")

## 12. Download results + wrappers


In [ ]:
import shutil
from google.colab import files
bundle = f"/content/eval_{LABEL}_{GIT_SHA}"
shutil.make_archive(bundle, "zip", EVAL_DIR, ".")   # results/, wrappers/, cases/
files.download(bundle + ".zip")